# Bekami — adaptateur RAG/agent (piste B)
Question posée : un adaptateur QLoRA entraîné sur des exemples *sources + question → réponse citée / refus / appel d'outil* améliore-t-il le comportement RAG et agent d'un petit modèle ouvert ?

**Attention.** Les données sont générées par gabarits (20 sujets, français). Un bon score au banc prouve que le modèle a appris le **format et la discipline de citation**, pas qu'il comprend des documents réels. Le banc (`data/eval/rag.jsonl`) n'a aucun prompt ni sujet en commun avec l'entraînement. Mesurez toujours le modèle de base d'abord : un adaptateur qui n'améliore pas ce score est un échec, même si la perte d'entraînement baisse.

Ici `evaluate_rag.py` n'exécute aucun code généré : pas besoin de session jetable.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Activez un GPU : Exécution > Modifier le type d\'exécution'
print(torch.cuda.get_device_name(0))

## 1. Projet
Sur votre Mac, depuis le dossier du projet : `zip -r ../bekami_llm_starter.zip . -x '*/__pycache__/*' '.git/*' 'data/raw/*' 'checkpoints/*'`, puis téléversez l'archive dans `/content`.

In [ ]:
!mkdir -p /content/bekami_llm && unzip -o -q /content/bekami_llm_starter.zip -d /content/bekami_llm
%cd /content/bekami_llm
!pip install -q -r requirements-ft.txt
# torchao préinstallé (trop ancien) fait planter peft ; le projet ne l'utilise pas
!pip uninstall -y -q torchao


## 2. Données et banc
`build_rag_sft.py` régénère `adapters/rag-agent/{train,val}.jsonl` et `data/eval/rag.jsonl` (déterministe) et ajoute vos exemples relus de `data/curated/`. Le banc doit afficher 100 % avec les réponses de référence.

In [ ]:
!python build_rag_sft.py --n-train 2000 --n-eval 240
!python evaluate_rag.py --oracle
!grep base_model adapters/rag-agent/adapter.yaml

## 3. Mesure de référence (modèle de base, avant entraînement)

In [ ]:
!python evaluate_rag.py --hf Qwen/Qwen2.5-1.5B-Instruct --out base_rag.json

## 4. Entraînement de l'adaptateur
Le modèle ci-dessus doit être le même que `base_model` dans `adapters/rag-agent/adapter.yaml`.

Colab coupe les sessions : les points de reprise sont écrits sur **Google Drive** toutes les 10 étapes. Si la session s'arrête, reconnectez-vous, relancez les cellules 1 à 3 (GPU, projet, données) puis cette cellule : l'entraînement repart du dernier point.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python finetune.py rag-agent --runs-dir /content/drive/MyDrive/bekami_rag-agent_runs

## 5. Mesure avec l'adaptateur

In [ ]:
!python evaluate_rag.py --hf Qwen/Qwen2.5-1.5B-Instruct --adapter rag-agent --out tuned_rag.json

## 6. Comparaison par comportement

In [ ]:
import json
base, tuned = json.load(open('base_rag.json')), json.load(open('tuned_rag.json'))
print(f"{'comportement':<14}{'base':>8}{'adaptateur':>12}")
for k in tuned['by_kind']:
    print(f"{k:<14}{base['by_kind'][k]:>8.0%}{tuned['by_kind'][k]:>12.0%}")
print(f"{'GLOBAL':<14}{base['score']:>8.0%}{tuned['score']:>12.0%}   ({tuned['n']} cas)")
print('Chaque type compte 8 à 40 cas : un écart de quelques points est dans le bruit.')

## 7. Sauvegarde de l'adaptateur final sur Drive

In [ ]:
!cp -r adapters/rag-agent/weights /content/drive/MyDrive/bekami_rag-agent_weights